<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkST0151ENSkillsNetwork20531532-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>



#### Import the required libraries we need for the lab.


In [1]:
import piplite
await piplite.install(['numpy'],['pandas'])
await piplite.install(['seaborn'])

In [2]:
import pandas as pd
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as pyplot
import scipy.stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

<ipython-input-2-b3fdaf15785b>:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


#### Read the dataset in the csv file from the URL


In [3]:
from js import fetch
import io

URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ST0151EN-SkillsNetwork/labs/boston_housing.csv'
resp = await fetch(URL)
boston_url = io.BytesIO((await resp.arrayBuffer()).to_py())

In [4]:
boston_df=pd.read_csv(boston_url)

#### Add your code below following the instructions given in the course to complete the peer graded assignment


In [ ]:
# Use the dataframe you already loaded from the URL
df = boston_df.copy()

# Quick sanity checks
print(df.shape)
display(df.head())
display(df.info())
display(df.describe())

# alpha for all tests
ALPHA = 0.05

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

medv = df['MEDV'].dropna()

plt.figure(figsize=(6,4))
plt.boxplot(medv, vert=True, labels=['MEDV'])
plt.title("Boxplot of Median Value of Owner-Occupied Homes (MEDV)")
plt.ylabel("MEDV ($1000's)")
plt.show()

q1, q2, q3 = np.percentile(medv, [25, 50, 75])
iqr = q3 - q1
lw, uw = q1 - 1.5*iqr, q3 + 1.5*iqr
outliers = ((medv < lw) | (medv > uw)).sum()

print(f"Explanation:")
print(f"- Median (Q2): {q2:.2f} (in $1000's)")
print(f"- IQR (Q3-Q1): {iqr:.2f}")
print(f"- Whisker approx. range: [{lw:.2f}, {uw:.2f}]")
print(f"- Outliers (1.5*IQR rule): {outliers}")

In [ ]:
chas_counts = df['CHAS'].value_counts().sort_index()
props = chas_counts / chas_counts.sum()

plt.figure(figsize=(6,4))
plt.bar(chas_counts.index.astype(str), chas_counts.values)
plt.title("Bar Plot of Charles River Variable (CHAS)")
plt.xlabel("Tract Bounds Charles River (1=Yes, 0=No)")
plt.ylabel("Count")

# value labels
for x, y in zip(chas_counts.index.astype(str), chas_counts.values):
    plt.text(int(x)-0.05, y+0.5, str(y))

plt.show()

print("Explanation:")
for k in chas_counts.index:
    print(f"- CHAS={k}: {chas_counts[k]} tracts ({props[k]*100:.1f}%)")

In [ ]:
age_bins   = [0, 35, 70, np.inf]
age_labels = ['<=35 years', '35-70 years', '>=70 years']
df['AGE_group'] = pd.cut(df['AGE'], bins=age_bins, labels=age_labels, right=True, include_lowest=True)

groups = [df.loc[df['AGE_group'] == lab, 'MEDV'].dropna() for lab in age_labels]

plt.figure(figsize=(8,5))
plt.boxplot(groups, labels=age_labels, showmeans=True)
plt.title("Median Home Value (MEDV) by Housing Age Group")
plt.xlabel("Age Group")
plt.ylabel("MEDV ($1000's)")
plt.show()

print("Explanation:")
for lab, g in zip(age_labels, groups):
    if len(g) > 0:
        print(f"- {lab}: n={len(g)}, median={np.median(g):.2f}, mean={np.mean(g):.2f}")

In [ ]:
from scipy import stats

xy = df[['INDUS','NOX']].dropna()
x, y = xy['INDUS'], xy['NOX']

plt.figure(figsize=(7,5))
plt.scatter(x, y, alpha=0.7)
plt.title("Relationship: NOX vs INDUS")
plt.xlabel("Proportion of Non-Retail Business Acres (INDUS)")
plt.ylabel("Nitric Oxides Concentration (parts per 10 million)")
plt.show()

r, p = stats.pearsonr(x, y)
print(f"Explanation:")
print(f"- Pearson r = {r:.3f}, p = {p:.4f}")
print("- Positive r suggests higher industrial land share associates with higher NOX.")

In [ ]:
ptr = df['PTRATIO'].dropna()

plt.figure(figsize=(7,5))
plt.hist(ptr, bins=15, edgecolor='black')
plt.title("Histogram of Pupil–Teacher Ratio (PTRATIO)")
plt.xlabel("Pupil–Teacher Ratio")
plt.ylabel("Frequency")
plt.show()

print(f"Explanation:")
print(f"- Mean: {ptr.mean():.2f}, Median: {np.median(ptr):.2f}, Std: {ptr.std(ddof=1):.2f}")

In [ ]:
medv_chas1 = df.loc[df['CHAS']==1, 'MEDV'].dropna()
medv_chas0 = df.loc[df['CHAS']==0, 'MEDV'].dropna()

t_stat, p_val = stats.ttest_ind(medv_chas1, medv_chas0, equal_var=False)

print("Hypotheses:")
print("H0: Mean(MEDV|CHAS=1) = Mean(MEDV|CHAS=0)")
print("H1: Means differ")
print(f"\nTest statistic t = {t_stat:.3f}, p = {p_val:.4f}")

if p_val < ALPHA:
    print(f"Decision @ α={ALPHA}: Reject H0 — significant difference in MEDV by Charles River adjacency.")
else:
    print(f"Decision @ α={ALPHA}: Fail to Reject H0 — no significant difference detected.")

In [ ]:
g1 = df.loc[df['AGE_group']=='<=35 years', 'MEDV'].dropna()
g2 = df.loc[df['AGE_group']=='35-70 years', 'MEDV'].dropna()
g3 = df.loc[df['AGE_group']=='>=70 years', 'MEDV'].dropna()

f_stat, p_val = stats.f_oneway(g1, g2, g3)

print("Hypotheses:")
print("H0: All AGE groups have the same mean MEDV")
print("H1: At least one group mean differs")
print(f"\nF = {f_stat:.3f}, p = {p_val:.4f}")

if p_val < ALPHA:
    print(f"Decision @ α={ALPHA}: Reject H0 — MEDV differs across AGE groups.")
    print("Note: For which groups differ, run post-hoc comparisons (e.g., Tukey) if available.")
else:
    print(f"Decision @ α={ALPHA}: Fail to Reject H0 — no evidence of a difference among AGE groups.")


Pearson correlation: NOX vs INDUS

In [ ]:
r, p_val = stats.pearsonr(df['NOX'].dropna(), df['INDUS'].dropna())  # assumes aligned, else use xy above
print("Hypotheses:")
print("H0: ρ = 0 (no linear relationship)")
print("H1: ρ ≠ 0")
print(f"\nPearson r = {r:.3f}, p = {p_val:.4f}")

if p_val < ALPHA:
    print(f"Decision @ α={ALPHA}: Reject H0 — significant linear relationship between NOX and INDUS.")
else:
    print(f"Decision @ α={ALPHA}: Fail to Reject H0 — no significant linear relationship detected.")

2D) Regression: Effect of DIS on MEDV (simple linear)

In [ ]:
res = stats.linregress(df['DIS'], df['MEDV'])

print("Hypotheses:")
print("H0: β(DIS) = 0 (no effect on MEDV)")
print("H1: β(DIS) ≠ 0")
print(f"""
Slope β(DIS): {res.slope:.3f}  (Δ MEDV in $1000's per +1 DIS)
Intercept   : {res.intercept:.3f}
t-statistic : {res.slope/res.stderr:.3f}
p-value     : {res.pvalue:.4f}
R-squared   : {res.rvalue**2:.3f}
""")

if res.pvalue < ALPHA:
    print(f"Decision @ α={ALPHA}: Reject H0 — DIS has a statistically significant association with MEDV.")
else:
    print(f"Decision @ α={ALPHA}: Fail to Reject H0 — no significant association detected.")

# Visualization with fitted line
x = df['DIS'].values
y = df['MEDV'].values
y_hat = res.intercept + res.slope * x

plt.figure(figsize=(7,5))
plt.scatter(x, y, alpha=0.7, label="Data")
plt.plot(x, y_hat, linewidth=2, label="Fitted line")
plt.title("Impact of Distance to Employment Centers (DIS) on MEDV")
plt.xlabel("Weighted Distance to Employment Centers (DIS)")
plt.ylabel("MEDV ($1000's)")
plt.legend()
plt.show()